In [1]:
### DATA INGESTION to vector DB Pipeline


In [3]:
import os 
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path


In [4]:
## read all the pdf inside the directory
 

def process_pdf_directory(pdf_directory):
    """Process all PDF files in a directory"""
    
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
        
        except Exception as e:
            print(f"  ✗ Error loading {pdf_file.name}: {e}")
    
    return all_documents


# ── Usage ──
all_docs = process_pdf_directory("../data/")
print(f"\nTotal pages loaded: {len(all_docs)}")

Found 4 PDF files to process

Processing: attention.pdf
  ✓ Loaded 3 pages

Processing: emneddings.pdf
  ✓ Loaded 3 pages

Processing: objectdetection.pdf
  ✓ Loaded 3 pages

Processing: proposal.pdf
  ✓ Loaded 3 pages

Total pages loaded: 12


In [5]:
all_docs

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-04-01T11:25:58+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-04-01T11:25:58+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\attention.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'attention.pdf', 'file_type': 'pdf'}, page_content="Attention Is All You Need\n Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit,\n Llion Jones, Aidan N. Gomez, Lukasz Kaiser, Illia Polosukhin\n Google Brain   |   Google Research   |   University of Toronto\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or convolutional\nneural networks that include an encoder and a decoder. The best performing models also\nconnect the encoder and decoder through an attention mechanism. We propose a new simple\nnetwork architecture, the Transformer, based sole

In [6]:
### text splitting chunks 
def split_documents(documents):
    """Split documents into smaller chunks"""
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs
    


In [8]:
chunks=split_documents(all_docs)
chunks

Split 12 documents into 27 chunks

Example chunk:
Content: Attention Is All You Need
 Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit,
 Llion Jones, Aidan N. Gomez, Lukasz Kaiser, Illia Polosukhin
 Google Brain   |   Google Research   |   Universit...
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-04-01T11:25:58+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-04-01T11:25:58+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\attention.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'attention.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-04-01T11:25:58+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-04-01T11:25:58+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\attention.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'attention.pdf', 'file_type': 'pdf'}, page_content='Attention Is All You Need\n Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit,\n Llion Jones, Aidan N. Gomez, Lukasz Kaiser, Illia Polosukhin\n Google Brain   |   Google Research   |   University of Toronto\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or convolutional\nneural networks that include an encoder and a decoder. The best performing models also\nconnect the encoder and decoder through an attention mechanism. We propose a new simple\nnetwork architecture, the Transformer, based sole

In [9]:
### Emnbedding And vectoreStoreDB
 
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

d:\anaconda3\envs\legal-doc-analyzer\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
import numpy as np
from typing import List
from sentence_transformers import SentenceTransformer

class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


# Initialize the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3685.93it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384
